# 04 — Training and Forecasting

**Purpose:** load the fitted model artifacts, generate current lifecycle forecasts, build the 14-day → 7-day holdout comparison, and export stakeholder-facing outputs. Training logic itself remains in the modeling module.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\BaneleS\Downloads\ml_refactor_final


In [2]:
import pandas as pd

from game_performance.config import SETTINGS
from game_performance.io import load_artifact, load_dataframe, save_dataframe
from game_performance.reporting import build_holdout_comparison, export_excel
from game_performance.scoring import run_daily_lifecycle_scoring

SETTINGS.ensure_directories()

df = load_dataframe(SETTINGS.processed_dir / "clean_game_daily.pkl")
training_results = load_artifact(SETTINGS.model_dir / "training_results.joblib")
model_bundles = training_results["model_bundles"]

print("Loaded cleaned data:", df.shape)
print("Loaded model families:", len(model_bundles))


Loaded cleaned data: (880049, 13)
Loaded model families: 10


In [3]:
current_predictions, scoring_log = run_daily_lifecycle_scoring(
    df,
    model_bundles,
    SETTINGS.horizon_strategy,
    as_of_date=df["Date"].max(),
    exact_checkpoint_only=False,
)

print(f"Generated {len(current_predictions):,} predictions")
display(current_predictions.head(30))

if scoring_log is not None and len(scoring_log): 
    print("Scoring log")
    display(scoring_log)


Generated 41,758 predictions


,Game_ID,TournamentName,SystemName,First_Observed_Date,Lifecycle_Stage,Obs_Window,Horizon,active_observation_days,observation_coverage,predicted_clients,...,predicted_stakes,predicted_ggr,predicted_ggr_pct,probability_high_performance,predicted_high_performer,derived_predicted_ggr_pct,recommendation,scored_at,Checkpoint,As_Of_Date
0,Erlang-Online::AFC Futsal Championship,AFC Futsal Championship,Erlang-Online,2026-02-01,Day 7,7,7,4,0.571429,38.686303,...,0.000000,422.479178,0.418564,0.009895,0,NaN,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
1,"Erlang-Online::ATP Challenger Brisbane, Austra...","ATP Challenger Brisbane, Australia Men Singles",Erlang-Online,2026-02-01,Day 7,7,7,7,1.000000,227.205259,...,236891.159496,11466.465967,0.126331,0.329435,0,0.048404,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
2,"Erlang-Online::ATP Challenger Cesenatico, Ital...","ATP Challenger Cesenatico, Italy Men Singles",Erlang-Online,2026-02-01,Day 7,7,7,7,1.000000,325.403556,...,16214.885880,1982.104517,0.254716,0.019499,0,0.122240,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
3,"Erlang-Online::Atp Challenger Cleveland, Usa M...","Atp Challenger Cleveland, Usa Men Singles",Erlang-Online,2026-02-01,Day 7,7,7,7,1.000000,234.280804,...,65763.625147,5234.564766,0.125664,0.062354,0,0.079597,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
4,"Erlang-Online::ATP Challenger Koblenz, Germany...","ATP Challenger Koblenz, Germany Men Singles",Erlang-Online,2026-02-01,Day 7,7,7,7,1.000000,230.934308,...,118864.445304,8003.697257,0.122895,0.091688,0,0.067335,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
5,"Erlang-Online::ATP Challenger Rosario, Argenti...","ATP Challenger Rosario, Argentina Men Singles",Erlang-Online,2026-02-01,Day 7,7,7,7,1.000000,253.223654,...,25329.757700,2110.322430,0.165304,0.019045,0,0.083314,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
6,"Erlang-Online::Atp Challenger Tenerife, Spain ...","Atp Challenger Tenerife, Spain Men Singles",Erlang-Online,2026-02-01,Day 7,7,7,7,1.000000,248.309920,...,66615.645197,2374.766658,0.130398,0.047893,0,0.035649,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
7,"Erlang-Online::Atp Montpellier, France Men Sin...","Atp Montpellier, France Men Singles",Erlang-Online,2026-02-01,Day 7,7,7,7,1.000000,2083.041337,...,338504.606783,16146.388119,0.126035,0.467375,0,0.047699,Standard placement with monitoring,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
8,Erlang-Online::India RFDL U21,India RFDL U21,Erlang-Online,2026-02-01,Day 7,7,7,2,0.285714,39.358881,...,4038.524865,348.079438,0.230933,0.015280,0,0.086190,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29
9,Erlang-Online::Indonesia Liga 3,Indonesia Liga 3,Erlang-Online,2026-02-01,Day 7,7,7,4,0.571429,88.888417,...,4759.399562,1152.279346,0.295327,0.018014,0,0.242106,Standard catalogue placement,2026-08-19 13:38:34.939604+00:00,7,2026-07-29


Scoring log


,Game_ID,Checkpoint,Horizon,As_Of_Date,scored_at
0,Erlang-Online::AFC Futsal Championship,7,7,2026-07-29,2026-08-19 13:38:34.939604+00:00
1,"Erlang-Online::ATP Challenger Brisbane, Austra...",7,7,2026-07-29,2026-08-19 13:38:34.939604+00:00
2,"Erlang-Online::ATP Challenger Cesenatico, Ital...",7,7,2026-07-29,2026-08-19 13:38:34.939604+00:00
3,"Erlang-Online::Atp Challenger Cleveland, Usa M...",7,7,2026-07-29,2026-08-19 13:38:34.939604+00:00
4,"Erlang-Online::ATP Challenger Koblenz, Germany...",7,7,2026-07-29,2026-08-19 13:38:34.939604+00:00
...,...,...,...,...,...
41753,Gaming Realms::SLO-Slingo Loteria Don Clemente,21,60,2026-07-29,2026-08-19 13:45:22.006686+00:00
41754,Gaming Realms::SLO-Slingo Rakin Bacon,21,60,2026-07-29,2026-08-19 13:45:22.006686+00:00
41755,Gaming Realms::SLO-Slingo Xxxtreme Express,21,60,2026-07-29,2026-08-19 13:45:22.006686+00:00
41756,GgamingCrash::GGC-Crash,21,60,2026-07-29,2026-08-19 13:45:22.006686+00:00


In [4]:
holdout = training_results["holdout_predictions"]

print("Holdout predictions:", holdout.shape)

if holdout.empty:
    raise ValueError(
        "No holdout predictions are available. Run 03_model_experiments_production.ipynb first."
    )

available = (
    holdout[["Obs_Window", "Horizon"]]
    .drop_duplicates()
    .sort_values(["Obs_Window", "Horizon"])
    .reset_index(drop=True)
)

print("Available holdout model families:")
display(available)

available_pairs = {
    (int(row.Obs_Window), int(row.Horizon))
    for row in available.itertuples(index=False)
}

# Production preference. If full production training was run, this should exist.
preferred_pair = (14, 7)

if preferred_pair in available_pairs:
    checkpoint, horizon = preferred_pair
else:
    # Keep the notebook runnable, but make the issue obvious.
    checkpoint, horizon = tuple(available.iloc[-1].astype(int))
    print(
        f"WARNING: preferred production pair {preferred_pair} is not available. "
        f"Using checkpoint={checkpoint}, horizon={horizon} instead. "
        "Check skipped_configurations from notebook 03."
    )

comparison = build_holdout_comparison(
    holdout,
    checkpoint=int(checkpoint),
    horizon=int(horizon),
)

print(f"Comparison rows: {len(comparison):,}")
display(comparison.head(20))


Holdout predictions: (3165, 20)
Available holdout model families:


,Obs_Window,Horizon
0,7,7
1,7,14
2,7,30
3,14,7
4,14,14
5,14,30
6,14,60
7,21,14
8,21,30
9,21,60


Comparison rows: 369


,Game_ID,TournamentName,SystemName,First_Observed_Date,Observation_Start,Observation_End,Prediction_Start,Prediction_End,Obs_Window,Horizon,...,variance_stakes,probability_stakes,actual_ggr,predicted_ggr,variance_ggr,probability_ggr,actual_ggr_pct,predicted_ggr_pct,variance_ggr_pct,probability_ggr_pct
1164,"Erlang-Online::Western Australia Division 1, W...","Western Australia Division 1, Women",Erlang-Online,2026-06-14,2026-06-14,2026-06-27,2026-06-28,2026-07-04,14,7,...,-1.121722e+04,0.581511,5751.7684,1413.976558,-4337.791842,0.245833,0.214585,0.145930,-0.068655,0.680056
1165,Evolution::Evo-Lightning Baccarat,Evo-Lightning Baccarat,Evolution,2026-06-14,2026-06-14,2026-06-27,2026-06-28,2026-07-04,14,7,...,-4.008000e+04,0.000000,-9477.0000,3832.100047,13309.100047,0.000000,-0.236452,0.034696,0.271148,0.000000
1166,SyX-Retail::Argentina Regional League,Argentina Regional League,SyX-Retail,2026-06-14,2026-06-14,2026-06-27,2026-06-28,2026-07-04,14,7,...,9.766151e+02,0.000000,205.0000,1163.303951,958.303951,0.000000,1.000000,1.001110,0.001110,0.998890
1167,"Erlang-Online::MFF League, Women","MFF League, Women",Erlang-Online,2026-06-15,2026-06-15,2026-06-28,2026-06-29,2026-07-05,14,7,...,-9.927996e+04,0.398972,3240.3236,14467.381888,11227.058288,0.000000,0.019616,0.185722,0.166105,0.000000
1168,Evolution::Evo-Blackjack VIP Alpha,Evo-Blackjack VIP Alpha,Evolution,2026-06-15,2026-06-15,2026-06-28,2026-06-29,2026-07-05,14,7,...,1.021514e+04,0.750241,5547.0000,5448.614258,-98.385742,0.982263,0.135623,-0.001293,-0.136917,0.000000
1169,SyX-Online::EMERALD DOWNS,EMERALD DOWNS,SyX-Online,2026-06-15,2026-06-15,2026-06-28,2026-06-29,2026-07-05,14,7,...,-2.669958e+03,0.515884,3877.5474,585.296232,-3292.251168,0.150945,0.703076,0.108827,-0.594249,0.154787
1170,SyX-Retail::Iceland League U20 Women,Iceland League U20 Women,SyX-Retail,2026-06-16,2026-06-16,2026-06-29,2026-06-30,2026-07-06,14,7,...,5.083873e+02,0.000000,-1997.9650,-418.420263,1579.544737,0.209423,-6.147585,0.904578,7.052162,0.000000
1171,PlayTechSlots::PlayTechS-Goal Goal Goal: Cash ...,PlayTechS-Goal Goal Goal: Cash Collect,PlayTechSlots,2026-06-17,2026-06-17,2026-06-30,2026-07-01,2026-07-07,14,7,...,3.108378e+05,0.865519,66164.6500,83693.807313,17529.157313,0.735068,0.028625,0.046761,0.018136,0.366454
1172,Jika-Online::Kiron-Rodeo,Kiron-Rodeo,Jika-Online,2026-06-17,2026-06-17,2026-06-30,2026-07-01,2026-07-07,14,7,...,3.494028e+04,0.784434,17807.7600,13290.194463,-4517.565537,0.746315,0.109866,0.097507,-0.012359,0.887510
1173,HackSawSlots::HSS-Ze Zeus,HSS-Ze Zeus,HackSawSlots,2026-06-17,2026-06-17,2026-06-30,2026-07-01,2026-07-07,14,7,...,-5.125382e+04,0.934752,45685.7000,27573.891386,-18111.808614,0.603556,0.058159,0.061371,0.003211,0.944785


In [5]:
from datetime import datetime
from pathlib import Path


def export_production_excel(
    path: Path,
    *,
    comparison,
    forecasts,
    regression,
    classification,
) -> Path:
    """Export the production report, using a timestamped copy if Excel has it open."""
    path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with open(path, "a"):
            pass
    except PermissionError:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = path.with_name(f"{path.stem}_{timestamp}{path.suffix}")
        print("Report is currently open.\n" f"Saving instead to:\n{path}")

    return export_excel(
        path,
        comparison=comparison,
        forecasts=forecasts,
        regression=regression,
        classification=classification,
    )


In [6]:
save_dataframe(current_predictions, SETTINGS.report_dir / "current_predictions.pkl")

report_path = SETTINGS.report_dir / "model_business_report.xlsx"

export_production_excel(
    report_path,
    comparison=comparison,
    forecasts=current_predictions,
    regression=training_results["regression_evaluation"],
    classification=training_results["classification_evaluation"],
)

print("Saved current predictions:", SETTINGS.report_dir / "current_predictions.pkl")
print("Saved Excel report:", report_path)


Saved current predictions: C:\Users\BaneleS\Downloads\ml_refactor_final\artifacts\reports\current_predictions.pkl
Saved Excel report: C:\Users\BaneleS\Downloads\ml_refactor_final\artifacts\reports\model_business_report.xlsx
